In [2]:
import os
import requests
import tiktoken
import numpy as np
import random
import torch
from torch import nn
import math

In [21]:
input_file_path = './data/tinyshakespeare/input.txt'

with open(input_file_path, 'r', encoding='utf-8') as f:
    data = f.read()
n = len(data)
train_data = data[:int(n*0.9)]
val_data = data[int(n*0.9):]

enc = tiktoken.get_encoding('gpt2')
train_ids = torch.tensor(enc.encode_ordinary(train_data), dtype=torch.long)
val_ids = torch.tensor(enc.encode_ordinary(val_data), dtype=torch.long)
print(f"train tokens: {len(train_ids):,}")
print(f"val tokens: {len(val_ids):,}")

train tokens: 301,966
val tokens: 36,059


In [48]:
class CausalSelfAttention(nn.Module):
    def __init__(self, T, d_m, h):
        super().__init__()
        self.d_k = int(d_m / h)
        self.d_v = int(d_m / h)

        self.W_Q = torch.randn((d_m, self.d_k * h))
        self.W_K = torch.randn((d_m, self.d_k * h))
        self.W_V = torch.randn((d_m, self.d_v * h))
        self.M = torch.triu(torch.ones(T, T) * -torch.inf, diagonal=1)
        self.W_O = torch.randn((h * self.d_v, d_m)) / math.sqrt(d_m)


    def forward(self, x):
        Q = x @ self.W_Q
        K = x @ self.W_K
        V = x @ self.W_V
        # TODO: split into the heads + batch
        S = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        P = torch.softmax(S + self.M, dim=-1)
        O = P @ V
        out = O @ self.W_O
        return out

In [ ]:
class ToyGPT(nn.Module):
    def __init__(self, T, vocab_size, L, h, d_m, d_ff):
        super().__init__()
        self.E = torch.randn((vocab_size, d_m))
        self.P = torch.randn((T, d_m))

        self.attn = CausalSelfAttention(T, d_m, h)
        self.layer_norm = nn.LayerNorm((T, d_m))
        self.l1 = nn.Linear(d_m, d_ff)
        self.l2 = nn.Linear(d_ff, d_m)
        self.relu = nn.ReLU()
        self.proj = nn.Linear(d_m, vocab_size)


    def forward(self, x):
        x_emb = self.E[x] + self.P

        # attention
        x_attn = self.attn(x_emb)

        # add + norm
        x_add1 = x_attn + x_emb
        x_norm1 = self.layer_norm(x_add1)

        # feed-forward
        x_ff = self.l2(self.relu(self.l1(x_norm1)))

        # add + norm
        x_add2 = x_ff + x_norm1
        x_norm2 = self.layer_norm(x_add2)

        logits = self.proj(x_norm2)

        return logits

In [63]:
T = block_size = 32
vocab_size = enc.n_vocab
L = n_layer = 3
h = n_head = 4
d_m = n_embd = 96
d_k = int(d_m / h)
d_v = int(d_m / h)
d_ff = 384  # 4 * d_m
B = batch_size = 8

In [ ]:
model = ToyGPT(T, vocab_size, L, h, d_m, d_ff)

In [65]:
i = random.randrange(len(train_ids) - T)
x_dummy = train_ids[i : i + T]

In [66]:
x_dummy

tensor([ 5253,   198,  4863,   465,  2081,    12,  1326,   415,  1486,    13,
        14438,   465,  1295,    11,   198,  1870,   351,  1336,  1627,   286,
          465,  4934,    11,   198, 29168,    82,  4453, 48886,    26,   257,
          582,  3025])

In [67]:
model(x_dummy).shape

torch.Size([32, 50257])